> ## SOLUTION / ANSWER KEY &mdash; Lab 8.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-customer-service-chatbot.ipynb`](../lab-12-capstone-customer-service-chatbot.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.12 &mdash; Capstone: The Multi-Agent Chatbot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Route a real ticket to real create_agent specialists that answer
- Run a VOTE when a charge is disputed; gate any refund on a human
- Assemble it into one process() and read a real end-to-end trace

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The multi-agent customer-service chatbot](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-12"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Capstone: the **multi-agent customer-service chatbot** (the client's Lab 4.2), end to end. A **supervisor**
routes each message to the matching **specialists** (the **real `create_agent`** agents from Lab 8.1/8.11);
each **really answers** over its own tool. When a charge is **disputed**, you don't paste a contradiction &mdash;
you run the **vote** from Lab 8.7 and **escalate a split** to a human. A **synthesiser** composes one reply,
and anything **irreversible** (a refund) is flagged **`needs_approval`** &mdash; never auto-done. Routing, the
vote, synthesis and the refund gate are rule-based (offline); the specialists run for real against Groq.

In [ ]:
# One process() ties it all together: route -> real specialists -> vote-on-dispute -> synthesise -> gate.
print("capstone flow: supervisor -> real specialists -> vote (on dispute) -> synthesise -> human-gate refund")

## Build it
Complete `build_specialist` (bind the tools), `decide` (the clear-majority test), `process` (collect each
engaged specialist's real finding), and the `needs_human` refund gate.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from collections import Counter

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

def build_specialist(tools, role):
    return create_agent(llm, tools, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")

def route(message):
    m = message.lower()
    kws = {"billing": ("charg", "refund", "invoice", "billed"), "tech": ("crash", "bug", "login", "broken")}
    hits = [name for name, ks in kws.items() if any(k in m for k in ks)]
    return hits if hits else ["general"]

def refund_intent(message):
    return any(k in message.lower() for k in ("refund", "charged twice", "duplicate", "dispute"))
def is_dispute(message):
    return "dispute" in message.lower()
def decide(verdicts, threshold=0.5):                    # Lab 8.7's vote
    top, n = Counter(verdicts).most_common(1)[0]
    if n / len(verdicts) > threshold:
        return {"decision": top, "escalate": False}
    return {"decision": None, "escalate": True}

def synthesize(findings):
    return " ".join(f"[{k}] {findings[k]}" for k in sorted(findings)) if findings else "forwarded to a human agent"

def specialist_reply(agent, message):
    result = agent.invoke({"messages": [("user", message)]}, config={"recursion_limit": 8})
    return result["messages"][-1].content

def process(message, agents):
    """Route -> run each REAL specialist -> vote on a dispute -> synthesise -> gate the refund."""
    involved = route(message)
    findings = {name: specialist_reply(agents[name], message) for name in involved if name in agents}
    conflict = None
    if "billing" in involved and is_dispute(message):
        conflict = decide(["refund", "deny"])          # a disputed charge -> reviewers SPLIT
        findings["billing"] = "reviewers split -> escalate to a human" if conflict["escalate"] else findings.get("billing", "")
    needs_human = ("billing" in involved and refund_intent(message)) or bool(conflict and conflict["escalate"])
    return {"agents": sorted(findings) or involved,
            "reply": synthesize(findings),
            "status": "needs_approval" if needs_human else "auto_ok",
            "conflict": bool(conflict)}

try:
    # Offline sanity of the deterministic team logic (no model call yet):
    print("route two-intent :", route("charged twice and the app keeps crashing"))
    print("route hours      :", route("what are your hours?"))
    print("dispute vote     :", decide(["refund", "deny"]))       # split -> escalate
    print("refund intent?   :", refund_intent("please refund me"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the real specialists once, then run **`process()`** on one representative two-intent ticket. It routes, invokes the real specialists, synthesises, and gates the refund &mdash; read the real trace.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        # Build the REAL specialists ONCE, then process ONE representative ticket end to end.
        AGENTS = {"billing": build_specialist([lookup_invoice], "billing"),
                  "tech":    build_specialist([known_issues], "tech")}

        msg = "I was charged twice for order 4471 and the app keeps crashing on login."
        print("route:", route(msg))
        print("\n-- one real specialist trace (billing) --")
        print_trace(AGENTS["billing"].invoke({"messages": [("user", msg)]}, config={"recursion_limit": 8}))

        out = process(msg, AGENTS)          # runs the engaged REAL specialists, then applies vote/synthesis/gate
        print("\nagents  :", out["agents"])
        print("status  :", out["status"], "| conflict:", out["conflict"])
        print("reply   :", out["reply"][:300])
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `process()` routed to **both** real specialists, each of which **called its own tool** (see the trace), then synthesised **one** grounded reply.
- The ticket carries a refund intent, so `status` is **`needs_approval`** &mdash; the irreversible action waits for a human, exactly as the deck demands.
- Swap in a message containing *"dispute"* and the **vote** fires: a split escalates &mdash; the team knows when *not* to decide alone.

## Your turn (open task &mdash; no grader)
Build a **suite** of tickets (a pure-tech one, a general *"what are your hours?"*, a *"dispute the charge on 4471"*)
and run `process()` over each &mdash; then check every outcome: only-tech is `auto_ok`, general falls back, and the
dispute both `conflict` and escalates to `needs_approval`. **What good looks like:** every ticket is routed to the
right specialists, refunds and splits are gated on a human, and no reply invents a fact its specialists didn't
find. (Each two-intent ticket makes several live calls &mdash; run the suite a few tickets at a time on the free tier.)

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
if groq_ready():
    AGENTS = {"billing": build_specialist([lookup_invoice], "billing"),
              "tech":    build_specialist([known_issues], "tech")}
    suite = ["the app keeps crashing on login",         # only tech -> auto_ok
             "what are your hours?",                     # general fallback
             "I want to dispute the charge on 4471"]     # dispute -> vote splits -> needs_approval
    for msg in suite:
        out = process(msg, AGENTS)
        print(f"{msg[:34]:36} -> status={out['status']:14} conflict={out['conflict']} agents={out['agents']}")
else:
    print("(add GROQ_API_KEY to .env)")

---
### Key takeaway
You built a multi-agent customer-service chatbot end to end -- real specialists coordinated by a supervisor, a vote when they conflict, findings synthesised into one reply, refunds gated on a human. That completes Day 4. Next: Day 5 -- agents in the real world, responsibly.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>